# Variational Monte Carlo in continuous space

## From Hamiltonian symmetries to a working bosonic Monte Carlo code

In this exercise we study $N$ identical bosons in one dimension. We first implement the essential continuous-space VMC ingredients ourselves, without NetKet. We then study a harmonic well, an anharmonic quartic well, and improved trial wavefunctions.

We use units $\hbar=m=1$ throughout. The required calculations concern non-interacting bosons; interactions and fermions are discussed as extensions at the end.

## 1. Particles, Hamiltonian symmetries, boundary conditions, and the VMC estimator

For non-interacting particles,

$$
H=\sum_{i=1}^N\left[-\frac12\frac{\partial^2}{\partial x_i^2}+V(x_i)\right].
$$

The particles move on the full real line, $x_i\in(-\infty,\infty)$. There are no hard-wall boundary conditions in the harmonic or quartic problems. The physical boundary condition is normalizability,
$$
\int d\mathbf{x}\,|\Psi(\mathbf{x})|^2=1,
$$
and the wavefunction must vanish sufficiently rapidly when any $|x_i|\to\infty$. For these smooth confining potentials, admissible wavefunctions are also smooth.

The labels $i=1,\ldots,N$ identify coordinates in the mathematical description. They do not necessarily identify experimentally distinguishable particles. Identical particles have the same mass and the same interactions with the external potential, so exchanging two of them cannot change a physical prediction. Quantum mechanics implements this indistinguishability through exchange symmetry:

$$\Psi(\ldots,x_i,\ldots,x_j,\ldots)=\begin{cases}
+\Psi(\ldots,x_j,\ldots,x_i,\ldots) & \text{for bosons},\\
-\Psi(\ldots,x_j,\ldots,x_i,\ldots) & \text{for fermions}.
\end{cases}$$

Here we study bosons. The plus sign means that the wavefunction is unchanged when two particles exchange positions. Consequently, a product of identical one-body orbitals, $\Psi(\mathbf{x})=\prod_i\phi(x_i)$, is already symmetric and is a natural starting point for non-interacting bosons. Exchange effects become non-trivial when interactions or fermionic statistics change the required symmetry.

If the external potential is even, $V(-x)=V(x)$, the one-body Hamiltonian commutes with parity, $(P\psi)(x)=\psi(-x)$, so $[h,P]=0$. The many-body global parity operation is $(P\Psi)(x_1,\ldots,x_N)=\Psi(-x_1,\ldots,-x_N)$, and $[H,P]=0$ as well. Eigenstates can therefore be chosen even or odd. For these confining one-dimensional problems the ground state is non-degenerate and can be chosen even and positive: the lowest state has no nodes, and sign changes can only add kinetic energy. This is why the ground-state ansatz below uses even functions and a real log amplitude.

We represent the wavefunction through $L(\mathbf{x})=\log\Psi(\mathbf{x})$. Show that the local energy can be written as

$$
E_{\rm loc}(\mathbf{x})=\sum_i\left[-\frac12\left(\partial_i^2L+(\partial_iL)^2\right)+V(x_i)\right].
$$

This identity is the bridge between the physics and the implementation: each ansatz must provide its logarithm and its first two coordinate derivatives. Products become sums in $L$, normalization constants disappear, and wavefunction ratios become exponentials of log differences.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import typing

rng = np.random.default_rng(1234)

## 2. Continuous-space Metropolis sampling

The probability density sampled by VMC is $p(\mathbf{x})\propto|\Psi(\mathbf{x})|^2$. Propose a new configuration by changing one randomly chosen coordinate,

$$
x_i' = x_i + \delta, \qquad \delta\sim U[-\Delta,\Delta].
$$

Because this proposal is symmetric, the Metropolis acceptance probability is $\min(1,|\Psi(\mathbf{x}')|^2/|\Psi(\mathbf{x})|^2)$. Express this ratio using the logarithm of the wavefunction and implement the missing functions.

In [ ]:
def metropolis_step(logpsi, x, logpsi_x, step_size, rng=rng):
    # TODO propose one-coordinate displacement and accept or reject it
    if logpsi_x is None:
        # for the sampling we work with a single sample
        # but our ansatz only supports batches
        # so we add a dummy batch dimension here
        logpsi_x = logpsi( np.expand_dims(x, 0))[0]

    index = tuple(np.random.randint(s) for s in x.shape())
    x_prime = x
    x_prime[index] = x[index] + np.random.uniform(-step_size, step_size)
    logpsi_x_prime = logpsi(x_prime)
    R = np.exp(2*(logpsi_x_prime-logpsi_x))
    accept = R > np.random.rand()

    if accept:
        return x_prime, logpsi_x_prime
    else:
        return x, logpsi_x


def sample_mcmc(logpsi, n_particles, n_samples, n_discard, step_size, x0=None, rng=rng):
    # TODO return an array with shape (n_samples, n_particles)
    if x0 is None:
        rand = np.random.normal(loc=0, scale=1, size=n_particles)
        x0 = np.exp(rand)
    
    x = x0
    logpsi_x = None
    
    for i in range(n_discard):
        x, logpsi = metropolis_step(logpsi, x, logpsi_x, step_size)
    
    samples = []
    for i in range(n_samples):
        x, logpsi = metropolis_step(logpsi, x, logpsi_x, step_size)
        if i%step_size == 0:
            samples.append(x)
        
    return np.vstack(samples)

## 3. A reusable local-energy interface

Implement a function that receives $\log\Psi$, $\nabla\log\Psi$, and $\nabla^2\log\Psi$, together with a potential and a batch of configurations. Keep the particle dimension explicit. Check that the returned local-energy array has one value per configuration.

In [ ]:
def local_energy(logpsi, grad_logpsi, laplacian_logpsi, potential, x):
    # TODO support both per-particle potentials with shape (n_samples, n_particles)
    # and total potentials with shape (n_samples,)
    
    e_pos = -.5*(np.sum(laplacian_logpsi) + np.sum(grad_logpsi)**2)
    V = potential(x)
    if tuple(V.shape())[1] != 0:
        V = np.sum(V,axis=1)
    
    return e_pos + V 

def estimate_energy(eloc):
    # TODO return the mean and standard error of correlated samples
    mean = np.mean(eloc)
    dev = np.std(eloc)
    return mean, dev

## 4. Harmonic well

Take $V(x)=\frac12\omega^2x^2$ and use the permutation-symmetric, parity-even ansatz

$$
\log\Psi_\alpha(\mathbf{x})=-\frac{\alpha}{2}\sum_i x_i^2,\qquad \alpha>0.
$$

Derive $\partial_i\log\Psi$ and $\partial_i^2\log\Psi$. Implement them. The exact many-body ground-state energy is $E_0=N\omega/2$, and the optimum is $\alpha=\omega$.

In [ ]:
def logpsi_gaussian(params, x):
    # TODO params[0] is alpha; x has shape (n_samples, n_particles)
    logpsi = - .5 * params[0] * np.sum(np.power(x, 2) ,axis = 1)
    return logpsi
    

def grad_logpsi_gaussian(params, x):
    # TODO return shape (n_samples, n_particles)
    return - params[0]*x

def laplacian_logpsi_gaussian(params, x):
    # TODO return shape (n_samples, n_particles)
    N, M = x.shape()
    dims = np.ones(N,M)
    return -params[0]*dims

def harmonic_potential(x, omega):
    return 0.5 * omega**2 * x**2


## 5. Variational optimization

For a real positive ansatz define $O_k=\partial_{\theta_k}\log\Psi$. Derive and implement

$$
\frac{\partial E}{\partial\theta_k}=2\left(\langle E_{\rm loc}O_k\rangle-\langle E_{\rm loc}\rangle\langle O_k\rangle\right).
$$

For the one-parameter Gaussian, $O_\alpha=-\frac12\sum_i x_i^2$. Use gradient descent, while enforcing $\alpha>0$.

In [ ]:
def gaussian_parameter_gradient(params, x, eloc):
    # TODO implement the covariance estimator
    O = -.5*np.sum(np.power(x,2), axis=1)
    grad_E = 2* np.cov(eloc, O)
    return grad_E

def optimize_gaussian(omega, n_particles, n_steps=100, alpha0=0.5):
    # TODO sample, estimate energy, compute gradient, and update alpha
    lr = 0.01
    params = [alpha0]
    samples = sample_mcmc(logpsi_gaussian, n_particles, n_samples=1000, n_discard=150, step_size=4)
    eloc= local_energy(logpsi_gaussian, grad_logpsi_gaussian(params, x), laplacian_logpsi_gaussian(params,x), harmonic_potential, x)
    grad = gaussian_parameter_gradient(params, x, eloc)
    
    params -= lr*grad

## 6. Quartic anharmonic well

Now take $V(x)=\lambda x^4$. First reuse the Gaussian ansatz. Show that

$$
\frac{E(\alpha)}{N}=\frac{\alpha}{4}+\frac{3\lambda}{4\alpha^2},\qquad \alpha_{\rm opt}=(6\lambda)^{1/3}.
$$

There is no elementary closed-form formula for the quartic ground-state energy. However, the energy can be known to high accuracy by solving the one-dimensional Schrodinger equation numerically. For $\lambda=1$, a trusted numerical benchmark is $E_0^{(1)}\simeq0.667986$. Use the scaling $E_0^{(1)}(\lambda)=\lambda^{1/3}E_0^{(1)}(1)$ and compare it with VMC. Explain why this value is a numerical benchmark rather than an exact closed-form expression.

In [ ]:
def quartic_potential(x, lam):
    return lam * x**4

# TODO run VMC for the quartic potential and compare with the Gaussian
# variational prediction and the numerical benchmark.

## 7. Improved trial wavefunction

Use the symmetry-preserving ansatz

$$
\log\Psi_{a,b}(\mathbf{x})=-\sum_i\left(\frac{a}{2}x_i^2+b x_i^4\right).
$$

Explain why it is bosonic and parity even, and why normalizability requires $b>0$. Implement its coordinate derivatives and parameter derivatives. Optimize $a$ and $b$, then compare the result against the Gaussian and the numerical benchmark.

In [ ]:
def logpsi_quartic(params, x):
    # TODO implement the improved log wavefunction
    pass

def grad_logpsi_quartic(params, x):
    # TODO coordinate gradient
    pass

def laplacian_logpsi_quartic(params, x):
    # TODO second coordinate derivative
    pass

def parameter_loggrad_quartic(params, x):
    # TODO return shape (n_samples, 2)
    pass

## 8. Interacting bosons: a finite-range Gaussian repulsion

Go beyond independent particles by adding a smooth repulsion to the harmonic trap:

$$
H=\sum_i\left[-\frac12\partial_i^2+\frac12\omega^2x_i^2\right]+\sum_{i<j}U(x_i-x_j).
$$

Use the finite-range Gaussian interaction

$$
U(r)=g\exp\left[-\frac{r^2}{2\sigma^2}\right],\qquad g>0.
$$

This interaction is large when particles are close. It is finite-range and smooth, so it avoids the singular derivatives associated with a contact interaction. The interaction energy is one number per configuration:

$$
V_{\mathrm{int}}(\mathbf{x})=\sum_{i<j}U(x_i-x_j).
$$

The product Gaussian is still bosonic, but it cannot describe correlations efficiently. Use the symmetric Jastrow-improved ansatz

$$
\log\Psi(\mathbf{x})=-\frac{a}{2}\sum_i x_i^2-c\sum_{i<j}\exp\left[-\frac{(x_i-x_j)^2}{2\ell^2}\right].
$$

The pair sum is unchanged by exchanging particles. Implement the pair interaction, the Jastrow log wavefunction, and its first and second coordinate derivatives. Compare the product Gaussian and Jastrow energies as the repulsion strength $g$ is increased. Check that the pair-distance distribution is suppressed near zero.

In [ ]:
def gaussian_pair_interaction(x, interaction_strength, interaction_range):
    # TODO return the total pair interaction for each configuration
    pass

def interacting_harmonic_potential(x, omega, interaction_strength, interaction_range):
    # TODO add the harmonic trap energy and pair interaction energy
    pass

def logpsi_interacting(raw_params, x, jastrow_range):
    # TODO implement the symmetric one-body plus Jastrow log wavefunction
    pass

def grad_logpsi_interacting(raw_params, x, jastrow_range):
    # TODO return the first derivative with respect to every particle coordinate
    pass

def laplacian_logpsi_interacting(raw_params, x, jastrow_range):
    # TODO return the second derivative with respect to every particle coordinate
    pass

# TODO sample the interacting system and compare the product and Jastrow ansatzes.

In [ ]:
def one_body_density(samples, bins):
    # TODO return histogram centers and density normalized to one
    pass

def pair_distances(samples):
    # TODO return all absolute distances |x_i - x_j| with i < j
    pass

def mean_square_position(samples):
    # TODO estimate <x^2> per particle
    pass

# Suggested scan parameters. More samples reduce histogram noise.
# coupling_values = np.array([0.0, 0.25, 0.5, 1.0, 2.0])
# scan_sample_count = 30000
# scan_discard_count = 2000
# scan_step_size = 0.8
# scan_jastrow_strength = 0.2

# TODO sample each coupling value with a fixed Jastrow protocol.
# TODO create a 2x2 panel with density, pair-distance distribution,
# TODO cloud size <x^2>, and close-pair probability P(r < sigma).
# TODO explain how the plots demonstrate cloud broadening and pair suppression.

## 9. Ground-state analysis: four observables

Use four observables to interpret the sampled ground state.

1. **One-body density**

   $$
   \rho(x)=\frac{1}{N}\left\langle\sum_i\delta(x-x_i)\right\rangle.
   $$

   Show that for samples $\mathbf{x}^{(s)}\sim|\Psi|^2$,

   $$
   \rho(x)\approx\frac{1}{N_sN}\sum_{s=1}^{N_s}\sum_i\delta(x-x_i^{(s)}).
   $$

2. **Pair-distance distribution**

   For $r\geq0$, define

   $$
   P(r)=\frac{2}{N(N-1)}\left\langle\sum_{i<j}\delta\left(r-|x_i-x_j|\right)\right\rangle.
   $$

   Explain the normalization:

   $$
   \int_0^\infty P(r)\,dr=1.
   $$

   Then derive the Monte Carlo estimator

   $$
   P(r)\approx\frac{2}{N_sN(N-1)}\sum_{s=1}^{N_s}\sum_{i<j}\delta\left(r-|x_i^{(s)}-x_j^{(s)}|\right).
   $$

   Explain how replacing each delta function by a histogram bin leads to `np.histogram(..., density=True)`.

3. **Cloud size**

   $$
   \langle x^2\rangle=\frac{1}{N}\left\langle\sum_i x_i^2\right\rangle.
   $$

   Implement its sample average.

4. **Close-pair probability**

   $$
   P(r<\sigma)=\int_0^\sigma P(r)\,dr.
   $$

   Show that it is estimated by the fraction of sampled unordered pairs with $|x_i-x_j|<\sigma$.

Use these observables in the 2x2 panel. Explain why increasing repulsion should broaden $\rho(x)$ and suppress $P(r)$ near $r=0$.

## 10. Going further: fermions

Bosons are indistinguishable particles with symmetric wavefunctions. Fermions are also indistinguishable, but their wavefunctions are antisymmetric:

$$
\Psi_F(\ldots,x_i,\ldots,x_j,\ldots)=-\Psi_F(\ldots,x_j,\ldots,x_i,\ldots).
$$

Therefore two identical fermions at the same position must have zero amplitude. This exchange hole is a quantum-statistical effect, not necessarily a force. A Slater determinant, possibly multiplied by a symmetric Jastrow factor, is the standard starting point. Nodes and signs mean that a globally smooth real $\log\Psi$ is unavailable, so fermionic VMC requires a separate treatment of signs or phases and encounters the fermion sign problem.

## 11. Boundary conditions and numerical domains

The harmonic, quartic, and interacting problems considered here live physically on $\mathbb{R}^N$. Their boundary condition is decay and square-integrability, not periodicity or a prescribed value at a finite wall. Explain why the trial wavefunctions used above satisfy this condition.

A finite-difference calculation of the quartic benchmark replaces $\mathbb{R}$ by $[-L,L]$ and often imposes $\psi(-L)=\psi(L)=0$. This is an artificial numerical boundary, justified only when the wavefunction is already negligible at $\pm L$. Explain why one should increase $L$ and refine the grid until the energy is stable.

The smooth Gaussian interaction does not create a cusp at particle coincidence. For fermions, by contrast, antisymmetry forces the wavefunction to vanish when two identical fermions meet. Discuss these as different kinds of boundary or nodal conditions.